# Actor-Critic: a policy that acts and a value that judges

_Reinforcement Learning_

**Keep policy gradients' flexibility but kill their variance — let a learned critic score each action with a one-step TD error, and train both nets together.**

The existing mod-actor-critic lesson framed the advantage as $A = Q - V$. This
       lesson states the full mechanism the way the RL curriculum needs it &mdash; two learners
       that train together, and the one-step TD (Temporal Difference) error that links them.

       Two function approximators, usually neural nets:

---

This notebook is a practice scaffold for the **Actor-Critic: a policy that acts and a value that judges** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch

In [ ]:
# !pip install gymnasium torch
import gymnasium as gym
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np

torch.manual_seed(0); np.random.seed(0)
env = gym.make("CartPole-v1")
nS = env.observation_space.shape[0]   # 4 state features
nA = env.action_space.n               # 2 actions: push left / right
gamma = 0.99

# ---- Shared-trunk actor-critic network ----
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(nS, 128), nn.Tanh())
        self.actor = nn.Linear(128, nA)   # logits -> softmax policy  pi_theta(a|s)
        self.critic = nn.Linear(128, 1)   # scalar value estimate     V_w(s)
    def forward(self, s):
        h = self.trunk(s)
        return self.actor(h), self.critic(h).squeeze(-1)

net = ActorCritic()
opt = torch.optim.Adam(net.parameters(), lr=3e-3)   # ONE optimizer, both heads
ENT_COEF, VAL_COEF, ROLLOUT = 0.01, 0.5, 32         # entropy bonus, critic weight, steps/update

def select(s):
    logits, value = net(torch.as_tensor(s, dtype=torch.float32))
    dist = torch.distributions.Categorical(logits=logits)
    a = dist.sample()
    return a.item(), dist.log_prob(a), dist.entropy(), value

returns_hist = []
for ep in range(400):
    s, _ = env.reset(seed=ep)
    done = False; ep_ret = 0.0
    logps, vals, rews, ents, masks = [], [], [], [], []
    while not done:
        a, logp, ent, v = select(s)
        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc
        logps.append(logp); vals.append(v); rews.append(r)
        ents.append(ent); masks.append(0.0 if done else 1.0)
        s = s2; ep_ret += r
        # update every ROLLOUT steps (or at episode end) -> "one step per few steps"
        if len(rews) == ROLLOUT or done:
            with torch.no_grad():
                # bootstrap the tail value: 0 if terminal, else V_w(s')
                _, boot = net(torch.as_tensor(s, dtype=torch.float32))
                boot = boot * (0.0 if done else 1.0)
            # discounted bootstrapped returns -> TD targets for the critic
            R, targets = boot, []
            for r_t, m in zip(reversed(rews), reversed(masks)):
                R = r_t + gamma * R * m
                targets.append(R)
            targets = torch.tensor(list(reversed(targets)), dtype=torch.float32)
            values  = torch.stack(vals)
            logp_t  = torch.stack(logps)
            ent_t   = torch.stack(ents)
            adv = (targets - values).detach()           # TD advantage A = target - V_w(s)
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)  # normalize -> stability
            actor_loss  = -(logp_t * adv).mean()        # -log pi * A
            critic_loss = F.mse_loss(values, targets)   # MSE on the TD target
            entropy     = ent_t.mean()
            loss = actor_loss + VAL_COEF * critic_loss - ENT_COEF * entropy
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            opt.step()
            logps, vals, rews, ents, masks = [], [], [], [], []
    returns_hist.append(ep_ret)
    if (ep + 1) % 50 == 0:
        print(f"ep {ep+1:4d}  avg return (last 50) = {np.mean(returns_hist[-50:]):6.1f}")

# CartPole is "solved" at ~475 average return; A2C climbs there in a few hundred eps.
print("final 50-ep average:", round(float(np.mean(returns_hist[-50:])), 1))

## Visualize the data & results

_Does the critic actually help? We pit two tabular policy-gradient agents on the SAME tiny slippery-corridor MDP — REINFORCE (Monte-Carlo full return, no critic) vs Actor-Critic (one-step TD-error advantage) — and plot return per episode. Actor-Critic should climb faster and sit higher with less wobble, because the critic's bootstrap cuts variance._

In [ ]:
import numpy as np

# A tiny SLIPPERY CORRIDOR MDP, built inline in numpy (no gym).
#   10 states in a line; state 9 = goal (+1, terminal). Start = state 0.
#   2 actions: 0=LEFT, 1=RIGHT. Intended move happens w.p. 0.85, else SLIP (stay).
#   Step cost -0.02 to reward speed.  gamma = 0.97.
nS, nA, GOAL = 10, 2, 9
gamma, STEP_COST, SLIP = 0.97, -0.02, 0.15

def step(s, a, rng):
    s2 = s if rng.random() < SLIP else (max(s-1,0) if a==0 else min(s+1,nS-1))
    if s2 == GOAL:  return s2, 1.0, True
    return s2, STEP_COST, False

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

def rollout(theta, rng, max_t=80):
    s, traj = 0, []
    for _ in range(max_t):
        a = rng.choice(nA, p=softmax(theta[s]))
        s2, r, done = step(s, a, rng); traj.append((s, a, r, s2, done)); s = s2
        if done: break
    return traj

# ---- REINFORCE: weight each step by the FULL Monte-Carlo return G_t (minus a mean baseline) ----
def train_reinforce(seed, episodes):
    rng = np.random.default_rng(seed); theta = np.zeros((nS, nA)); lr = 0.15; curve = []
    for _ in range(episodes):
        traj = rollout(theta, rng)
        G, Gs = 0.0, []
        for (s, a, r, s2, d) in reversed(traj):
            G = r + gamma * G; Gs.append(G)
        Gs = list(reversed(Gs)); b = np.mean(Gs)           # baseline
        for (s, a, r, s2, d), Gt in zip(traj, Gs):
            p = softmax(theta[s]); grad = -p; grad[a] += 1.0
            theta[s] += lr * (Gt - b) * grad               # weight = full return
        curve.append(sum(r for (_, _, r, _, _) in traj))
    return np.array(curve)

# ---- ACTOR-CRITIC: weight each step by the one-step TD error delta = r + gamma*V(s') - V(s) ----
def train_actor_critic(seed, episodes):
    rng = np.random.default_rng(seed)
    theta = np.zeros((nS, nA)); V = np.zeros(nS)           # actor + tabular critic
    lrA, lrC = 0.15, 0.2; curve = []
    for _ in range(episodes):
        s, total = 0, 0.0
        for _ in range(80):
            p = softmax(theta[s]); a = rng.choice(nA, p=p)
            s2, r, done = step(s, a, rng); total += r
            target = r + (0.0 if done else gamma * V[s2])
            delta = target - V[s]                          # TD error = advantage
            V[s] += lrC * delta                            # critic update
            grad = -p; grad[a] += 1.0
            theta[s] += lrA * delta * grad                 # actor update, weighted by delta
            s = s2
            if done: break
        curve.append(total)
    return np.array(curve)

EP, SEEDS = 400, range(12)
rc = np.mean([train_reinforce(s, EP)     for s in SEEDS], axis=0)
ac = np.mean([train_actor_critic(s, EP)  for s in SEEDS], axis=0)
smooth = lambda x, w=15: np.convolve(x, np.ones(w)/w, mode="valid")
rcs, acs = smooth(rc), smooth(ac)
idx = np.linspace(0, len(rcs)-1, 18).astype(int)
print("Actor-Critic:", [[int(i), round(float(acs[i]), 3)] for i in idx])
print("REINFORCE:   ", [[int(i), round(float(rcs[i]), 3)] for i in idx])
print("raw std last 150 -> AC:", round(float(np.std(ac[-150:])), 3),
      " REINFORCE:", round(float(np.std(rc[-150:])), 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A critic estimates $\hat V(s_t)=2.0$ and $\hat V(s_{t+1})=2.0$. The agent gets reward $r_t=-0.5$. With $\gamma=0.95$, compute the TD error $\delta_t$ and say which way the actor pushes action $a_t$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Form the bootstrapped target. — _The one-step action value estimate is $r_t+\gamma\hat V(s_{t+1}) = -0.5 + 0.95\times 2.0 = -0.5 + 1.9 = 1.4$._
- Subtract the current value. — _$\delta_t = 1.4 - 2.0 = -0.6$._
- Read the sign. — _$\delta_t \lt 0$: the action did worse than the critic expected, so the actor update $\nabla_\theta\log\pi\cdot\delta_t$ has a negative weight and pushes $a_t$'s probability down._

**Answer:** $\delta_t = (-0.5 + 0.95\cdot 2.0) - 2.0 = 1.4 - 2.0 = -0.6$. Negative, so the actor decreases the probability of $a_t$ — it underperformed the state's average. The critic also lowers $\hat V(s_t)$ toward $1.4$.

</details>

**Problem 2.** Why does the critic's baseline leave the policy gradient unbiased, yet still reduce its variance?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the baseline term. — _The extra term is $\mathbb{E}_a[\nabla_\theta\log\pi_\theta(a\mid s)\,b(s)]$, with $b(s)$ independent of the action $a$._
- Pull $b(s)$ out and simplify. — _It becomes $b(s)\sum_a\nabla_\theta\pi_\theta(a\mid s) = b(s)\,\nabla_\theta\sum_a\pi_\theta(a\mid s) = b(s)\,\nabla_\theta 1 = 0$ because probabilities sum to one._
- Interpret variance. — _The mean is unchanged (the term is zero), but subtracting $\hat V(s)$ recenters the weight from the raw return $G_t$ to the advantage $\delta_t$, whose magnitude — and thus the gradient's variance — is much smaller._

**Answer:** Because the baseline term has expectation exactly $0$ (probabilities sum to one, so $\nabla_\theta 1 = 0$), the mean gradient is untouched — unbiased. But replacing $G_t$ with the smaller-magnitude advantage $\delta_t$ shrinks the spread of the weight, so the variance drops. Same direction, less noise.

</details>

**Problem 3.** In Generalized Advantage Estimation, what do the extremes $\lambda=0$ and $\lambda=1$ correspond to, and which has more variance?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug in $\lambda=0$. — _$\hat A_t^{\text{GAE}(0)} = \sum_l (\gamma\cdot 0)^l\delta_{t+l} = \delta_t$ — just the one-step TD error._
- Plug in $\lambda=1$. — _$\hat A_t^{\text{GAE}(1)} = \sum_l \gamma^l\delta_{t+l}$, which telescopes to $G_t - \hat V(s_t)$ — the full Monte-Carlo advantage._
- Compare. — _$\lambda=1$ sums every future step's noise (high variance, but no bootstrap bias); $\lambda=0$ uses one step (low variance, but leans on the biased critic)._

**Answer:** $\lambda=0$ gives the one-step TD error $\delta_t$ (low variance, more bias from the critic); $\lambda=1$ gives the Monte-Carlo advantage $G_t-\hat V(s_t)$ (no bootstrap bias, but the highest variance). $\lambda=1$ has more variance. Intermediate $\lambda$ (e.g. $0.95$) blends the two.

</details>